# Día 8 · Unidad 7 (cierre) — pandas avanzado (I): duplicados con criterio, muestreo y `groupby`

**Objetivos de aprendizaje**
- Repasar y ampliar la detección de duplicados con `subset` y `keep`, y el conteo de valores únicos.
- Generar muestras aleatorias reproducibles con `.sample()`.
- Dominar `.groupby()`: el objeto `DataFrameGroupBy`, `.get_group()`, `.agg()` con varias agregaciones a la vez, y la iteración sobre grupos.

**Duración estimada:** 90 minutos.

**Requisitos previos:** Día 7 completo (pandas y NumPy — fundamentos).

**Nota:** este es el primer notebook del Día 8, que cierra la Unidad 7 (pandas avanzado) y cubre la Unidad 8 (visualización). El Día 7 ya introdujo duplicados y nulos a un nivel básico con `compras_seguros.csv`; aquí profundizamos, y añadimos lo que de verdad hace de pandas una herramienta de análisis: `groupby`.

In [ ]:
import numpy as np
import pandas as pd

cartera = pd.read_csv("../data/raw/cartera_polizas.csv", parse_dates=["fecha_alta", "fecha_baja"])
siniestros = pd.read_csv("../data/raw/siniestros.csv", parse_dates=["fecha_siniestro"])

print(cartera.shape, siniestros.shape)
siniestros.head()

## 1. Duplicados: más allá de `.duplicated()` a secas

En el Día 7 vimos `.duplicated()` sin parámetros (compara **todas** las columnas). En la práctica, casi siempre interesa un criterio más específico: ¿qué columnas definen que dos filas son "la misma"?

Los parámetros clave de [`.duplicated()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html) y [`.drop_duplicates()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html):

- `subset`: columnas sobre las que se evalúa el duplicado (por defecto, todas).
- `keep`: `"first"` (marca como duplicado todo salvo la primera aparición), `"last"` (salvo la última), o `False` (marca **todas** las ocurrencias, incluida la primera).

In [ ]:
# ¿Hay algún asegurado con más de una póliza? (no es un duplicado real, es información legítima)
print("Pólizas con asegurado_id repetido:", cartera.duplicated(subset=["asegurado_id"], keep=False).sum())

# Veamos un ejemplo concreto
un_asegurado_repetido = cartera[cartera.duplicated(subset=["asegurado_id"], keep=False)]["asegurado_id"].iloc[0]
cartera[cartera["asegurado_id"] == un_asegurado_repetido]

Esto **no** son duplicados a eliminar: un mismo asegurado puede tener varias pólizas distintas (una de auto, otra de hogar...). `subset` os deja decidir exactamente qué combinación de columnas define un duplicado real — aquí, sería `subset=["poliza_id"]` lo que de verdad indicaría un error de carga, no `asegurado_id` solo.

> ⚠️ **Error típico**: usar `.drop_duplicates()` sin pensar en `subset`, confiando en el comportamiento por defecto (todas las columnas). Si dos filas difieren en una sola columna sin importancia (por ejemplo, un campo de auditoría con la hora exacta de carga), `.drop_duplicates()` por defecto las consideraría filas distintas y **no** las eliminaría, aunque para el negocio sean el mismo registro repetido.

In [ ]:
print("Valores únicos de provincia:", cartera["provincia"].nunique())
print("Valores únicos de producto:", cartera["producto"].nunique(), "->", cartera["producto"].unique())

## 2. Muestreo aleatorio

`.sample()` extrae una muestra aleatoria de filas (o columnas, con `axis=1`). Es muy útil para revisar un dataset grande "a ojo" sin cargarlo entero, o para dividir datos en entrenamiento/prueba (lo veréis con más detalle en la Unidad 11).

In [ ]:
# Muestra del 1% de la cartera, fijando random_state para que sea reproducible
muestra = cartera.sample(frac=0.01, random_state=42)
print(f"Muestra: {len(muestra)} de {len(cartera)} filas")
muestra.head()

In [ ]:
# random_state fijo -> la misma muestra cada vez que se ejecuta
muestra_repetida = cartera.sample(frac=0.01, random_state=42)
assert muestra.index.equals(muestra_repetida.index), "Con el mismo random_state, la muestra debe ser idéntica"
print("Confirmado: misma semilla, misma muestra.")

> 📊 **En Excel esto sería...** como `ALEATORIO.ENTRE()` combinado con un filtro para quedarte con un subconjunto de filas al azar — con la ventaja de que `random_state` deja el resultado fijado y reproducible, en vez de cambiar cada vez que recalculas la hoja.

## 3. `groupby`: agrupar para agregar

`.groupby()` es, con diferencia, la herramienta más usada de pandas para resumir datos. La idea: dividir el DataFrame en grupos según el valor de una o más columnas, aplicar una función de agregación a cada grupo, y combinar los resultados.

> 📊 **En Excel esto sería...** exactamente una Tabla Dinámica: arrastras un campo al área de filas (el `by` de `groupby`) y otro al área de valores con una función de resumen (`mean`, `sum`, `count`...). `groupby` es lo mismo, con más control y reproducible en código.

In [ ]:
por_producto = siniestros.groupby("producto")
print(type(por_producto))

`.groupby()` no calcula nada todavía por sí solo — devuelve un objeto `DataFrameGroupBy`, una especie de índice adicional que le dice a pandas dónde mirar cuando le pidáis una operación de agregación. El atributo `.groups` muestra ese mapeo:

In [ ]:
list(por_producto.groups.keys())

### `.get_group()`

Para extraer un grupo concreto como su propio DataFrame:

In [ ]:
por_producto.get_group("AUTO").head()

### Agregaciones directas

Aplicar una función de agregación directamente sobre una columna del objeto agrupado:

In [ ]:
# Importe medio pagado por siniestro, por producto
por_producto["importe_pagado"].mean().round(2).sort_values(ascending=False)

In [ ]:
# Percentil 90 del importe pagado, por producto
por_producto["importe_pagado"].quantile(0.9).round(2).sort_values(ascending=False)

### `.agg()`: varias agregaciones a la vez

[`.agg()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html) permite combinar varias métricas en una sola llamada, con nombres de columna elegidos por vosotros — la forma más flexible y legible de resumir un `groupby`.

In [ ]:
resumen_siniestros = siniestros.groupby("producto").agg(
    importe_medio=("importe_pagado", "mean"),
    importe_total=("importe_pagado", "sum"),
    n_siniestros=("importe_pagado", "count"),
    importe_maximo=("importe_pagado", "max"),
).round(2).sort_values("importe_total", ascending=False)

resumen_siniestros

También podéis usar funciones propias (con `lambda` o una función con nombre) dentro de `.agg()`:

In [ ]:
resumen_por_estado = siniestros.groupby(["producto", "estado"]).agg(
    importe_medio=("importe_pagado", "mean"),
    n=("importe_pagado", "count"),
).round(2)

resumen_por_estado.head(10)

Agrupar por **varias columnas** (`["producto", "estado"]`) crea un índice jerárquico (`MultiIndex`) — cada combinación de valores de ambas columnas es un grupo propio. `.reset_index()` lo convierte de vuelta a columnas normales, si os resulta más cómodo trabajar así:

In [ ]:
resumen_por_estado.reset_index().head(10)

> ⚠️ **Error típico**: olvidar que, tras un `.groupby(...).agg(...)`, el resultado suele tener el campo agrupador como **índice**, no como columna — intentar filtrar después con `df["producto"] == "AUTO"` falla con `KeyError` porque `producto` ya no es una columna. La solución es `.reset_index()` (o usar `.loc["AUTO"]` para acceder por índice).

In [ ]:
try:
    resumen_siniestros[resumen_siniestros["producto"] == "AUTO"]
except KeyError as e:
    print("Error esperado:", e)

# Correcto: acceder por índice
print(resumen_siniestros.loc["AUTO"])

### Iteración sobre grupos

Cuando la lógica por grupo no encaja en una simple agregación (por ejemplo, "quedarme con los 2 siniestros de mayor importe **de cada producto**", no el máximo global), se puede iterar directamente sobre el objeto agrupado: cada vuelta del bucle da una tupla `(nombre_del_grupo, DataFrame_del_grupo)`.

In [ ]:
top2_por_producto = []

for nombre_grupo, datos_grupo in siniestros.groupby("producto"):
    top2_por_producto.append(datos_grupo.nlargest(2, "importe_pagado"))

top2_siniestros = pd.concat(top2_por_producto)
top2_siniestros[["siniestro_id", "producto", "tipo_siniestro", "importe_pagado"]]

`pd.concat()` combina la lista de DataFrames (uno por producto) en uno solo — lo veréis en detalle en el próximo notebook. Esto es exactamente lo que no se puede resolver con `.agg()` (que resume a un único número por grupo): aquí queremos **conservar filas completas**, las 2 de mayor importe de cada grupo, no un resumen agregado.

**Para ti:** con `cartera`, agrupa por `provincia` y calcula, con `.agg()`, el número de pólizas y la prima media de cada provincia, ordenado de mayor a menor prima media. ¿Qué provincia tiene la prima media más alta?

In [ ]:
# Tu código aquí


## Resumen

- `.duplicated()`/`.drop_duplicates()` con `subset` y `keep` permiten definir con precisión qué combinación de columnas identifica un duplicado real — sin `subset`, se comparan todas las columnas, lo que puede dejar pasar duplicados de negocio o marcar como duplicado algo que no lo es.
- `.sample(frac=...)` o `.sample(n=...)` extraen una muestra aleatoria; `random_state` la hace reproducible.
- `.groupby()` no calcula nada hasta que se le pide una agregación — divide el DataFrame en grupos y espera instrucciones.
- `.get_group()` extrae un grupo concreto como DataFrame; agregaciones directas (`.mean()`, `.sum()`...) resumen una columna por grupo.
- `.agg()` con argumentos con nombre (`nombre=("columna", "funcion")`) es la forma más flexible de combinar varias métricas en una sola llamada — y la que más veréis en código real de pandas.
- Tras un `.groupby(...).agg(...)`, la(s) columna(s) agrupadora(s) suelen quedar como **índice**, no como columna — `.reset_index()` las devuelve a columnas si hace falta filtrar o exportar después.
- Iterar sobre `groupby` (`for nombre, datos in df.groupby(...)`) es la vía cuando la lógica por grupo no es una simple agregación a un número, sino que necesita conservar filas completas (como "los 2 mayores de cada grupo").

**Siguiente:** `02_teoria_merge_concat_pivot_io.ipynb`, donde unimos `cartera` y `siniestros` con `.merge()`, construimos tablas dinámicas con `pivot_table`, y validamos una transformación de `groupby`+`merge` con IA.